# 모듈 6: 엔드 투 엔드 PyTorch 신경망 훈련 파이프라인

모듈 1~5에서 배운 모든 개념을 하나로 조립하여
실제로 데이터를 넣고 학습하고 평가하는 **완전한 훈련 루프**를 구축합니다.

**실습 1:** XOR 문제를 자동 학습으로 해결  
**실습 2:** 붓꽃(Iris) 데이터 3종 분류 파이프라인 완성

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)

---
## 실습 1: XOR 문제 자동 학습

모듈 2에서는 가중치를 직접 넣어줬지만,
이제는 **역전파 + 경사하강법**으로 기계가 스스로 정답 가중치를 찾도록 합니다.

### Step 1. 데이터 준비

In [ ]:
# XOR 입력과 정답
X_xor = torch.tensor([[0.0, 0.0],
                      [0.0, 1.0],
                      [1.0, 0.0],
                      [1.0, 1.0]])

y_xor = torch.tensor([[0.0],
                      [1.0],
                      [1.0],
                      [0.0]])

print(f"입력 shape: {X_xor.shape}, 정답 shape: {y_xor.shape}")

### Step 2. nn.Module로 신경망 클래스 정의

모듈 2의 순전파 흐름(①→②→③→④)을 `forward()` 메서드 안에 담습니다.

In [ ]:
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)   # 입력 2개 → 은닉 4개
        self.output = nn.Linear(4, 1)   # 은닉 4개 → 출력 1개
    
    def forward(self, x):
        # ① 은닉층 선형결합 + ② ReLU 활성화
        x = torch.relu(self.hidden(x))
        # ③ 출력층 선형결합 + ④ 시그모이드 활성화 (이진 확률)
        x = torch.sigmoid(self.output(x))
        return x

model_xor = XORNet()
print(model_xor)
print(f"\n학습 가능한 파라미터 수: {sum(p.numel() for p in model_xor.parameters())}")

### Step 3. 손실 함수 + 옵티마이저 선택

In [ ]:
criterion = nn.BCELoss()                                  # 모듈 3: 이진 분류 → BCE
optimizer = optim.Adam(model_xor.parameters(), lr=0.01)   # Adam: SGD의 개선 버전

### Step 4. 학습 루프 (5단계 템플릿)

In [ ]:
epochs = 1000
loss_history = []

for epoch in range(epochs):
    # ① Forward: 순전파 (모듈 2)
    predictions = model_xor(X_xor)
    
    # ② Loss: 손실 계산 (모듈 3)
    loss = criterion(predictions, y_xor)
    
    # ③ Zero Grad: 기울기 초기화 (누적 방지)
    optimizer.zero_grad()
    
    # ④ Backward: 역전파 - 연쇄 법칙으로 기울기 계산 (모듈 5)
    loss.backward()
    
    # ⑤ Step: 경사하강법으로 가중치 업데이트 (모듈 4)
    optimizer.step()
    
    loss_history.append(loss.item())
    
    if (epoch + 1) % 200 == 0:
        print(f"Epoch [{epoch+1:4d}/{epochs}] - Loss: {loss.item():.6f}")

print(f"\n최종 손실: {loss_history[-1]:.6f}")

### Step 5. 결과 확인 및 시각화

In [ ]:
# XOR 예측 결과 확인
with torch.no_grad():
    final_pred = model_xor(X_xor)
    print("=== XOR 자동 학습 최종 결과 ===")
    xor_targets = [0, 1, 1, 0]
    for x_val, pred, target in zip(X_xor, final_pred, xor_targets):
        decision = 1 if pred.item() > 0.5 else 0
        status = "정답" if decision == target else "오답"
        print(f"  입력 {x_val.tolist()} → 확률 {pred.item():.4f} → 판정 {decision} (정답 {target}) {status}")

# 손실 감소 그래프
plt.figure(figsize=(10, 4))
plt.plot(loss_history, color='steelblue')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('BCE Loss', fontsize=12)
plt.title('XOR 학습 과정: 에폭별 손실 감소', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("→ 모듈 2에서 수동으로 넣어줬던 가중치를, 역전파+경사하강법이 자동으로 발견했습니다!")

---
## 실습 2: 붓꽃(Iris) 3종 분류 파이프라인

실전 데이터셋(150개 샘플, 4개 특성, 3개 종)을 사용하여
DataLoader 미니배치 → 학습 → 평가까지 완전한 파이프라인을 구성합니다.

### Step 1. 데이터 준비 (sklearn 활용)

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 데이터 로드
iris = load_iris()
X_np, y_np = iris.data, iris.target
print(f"전체 데이터: {X_np.shape[0]}개, 특성 {X_np.shape[1]}개, 종류 {len(set(y_np))}개")
print(f"종 이름: {iris.target_names.tolist()}")

# 학습/테스트 분리 (80:20)
X_train, X_test, y_train, y_test = train_test_split(X_np, y_np, test_size=0.2, random_state=42)

# 정규화 (특성 스케일 통일)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 파이토치 텐서로 변환
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# DataLoader 구성 (미니배치 자동 관리)
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

print(f"\n학습 데이터: {len(X_train_t)}개, 테스트 데이터: {len(X_test_t)}개")
print(f"미니배치 크기: 16, 에폭당 반복 횟수: {len(train_loader)}")

### Step 2. 다중 분류 신경망 정의

In [ ]:
class IrisClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(4, 16)    # 입력 4개 특성 → 은닉 16개 뉴런
        self.layer2 = nn.Linear(16, 8)    # 은닉 16 → 은닉 8
        self.layer3 = nn.Linear(8, 3)     # 은닉 8 → 출력 3개 (3종 분류)
    
    def forward(self, x):
        x = torch.relu(self.layer1(x))    # 은닉층 1 (ReLU)
        x = torch.relu(self.layer2(x))    # 은닉층 2 (ReLU)
        x = self.layer3(x)                # 출력층 (softmax는 CrossEntropyLoss 내부에서 자동 적용)
        return x

model_iris = IrisClassifier()
print(model_iris)
print(f"\n총 파라미터 수: {sum(p.numel() for p in model_iris.parameters())}")

### Step 3. 손실 함수 + 옵티마이저

In [ ]:
# 다중 분류 → CrossEntropyLoss (모듈 3에서 배운 CE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_iris.parameters(), lr=0.01)

### Step 4. 학습 루프 (DataLoader + 5단계)

In [ ]:
epochs = 100
epoch_losses = []

for epoch in range(epochs):
    epoch_loss = 0.0
    
    for batch_X, batch_y in train_loader:     # DataLoader가 미니배치를 자동 제공
        # ① Forward
        predictions = model_iris(batch_X)
        # ② Loss
        loss = criterion(predictions, batch_y)
        # ③ Zero Grad
        optimizer.zero_grad()
        # ④ Backward
        loss.backward()
        # ⑤ Step
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    epoch_losses.append(avg_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1:3d}/{epochs}] - 평균 Loss: {avg_loss:.4f}")

### Step 5. 평가 (Evaluation)

In [ ]:
# 평가 모드: 기울기 계산 불필요 (메모리 절약)
with torch.no_grad():
    test_outputs = model_iris(X_test_t)
    _, predicted = torch.max(test_outputs, dim=1)  # 가장 높은 확률의 클래스 선택
    
    correct = (predicted == y_test_t).sum().item()
    total = y_test_t.size(0)
    accuracy = correct / total * 100

print("=" * 50)
print(f"테스트 정확도: {accuracy:.1f}% ({correct}/{total})")
print("=" * 50)

# 손실 감소 그래프
plt.figure(figsize=(10, 4))
plt.plot(epoch_losses, color='darkorange', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Average Loss', fontsize=12)
plt.title('Iris 분류 학습 과정: 에폭별 평균 손실 감소', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 최종 정리: 전체 커리큘럼 연결 지도

| 모듈 | 배운 것 | 파이프라인에서의 역할 |
|:---:|:---|:---|
| 모듈 1 | 퍼셉트론, 선형결합, 계단함수 | 신경망의 근본 원리 |
| 모듈 2 | 다층 구조, 순전파 흐름, Sigmoid/ReLU | `model.forward()` |
| 모듈 3 | MSE, BCE, CE 손실 함수 | `criterion(pred, target)` |
| 모듈 4 | 미분과 경사하강법 | `optimizer.step()` |
| 모듈 5 | 연쇄법칙과 오차역전파 | `loss.backward()` |
| **모듈 6** | **전체 조립: 파이프라인 완성** | **학습 루프 5단계** |

**축하합니다.** 인공신경망의 기초를 완주했습니다.

## 실전 체크 예제: 문제 유형별 공식 선택

파이프라인에서 가장 흔한 실수는 문제 유형과 손실/출력 공식을 잘못 매칭하는 것입니다.

- 이진 분류: `Sigmoid + BCE`
- 다중 분류: `Softmax/Logits + CE`
- 회귀: `Linear 출력 + MSE`


In [ ]:
mapping = [
    ('이진 분류', '출력층: Sigmoid', '손실: BCE'),
    ('다중 분류', '출력층: Logits(또는 Softmax)', '손실: CrossEntropy'),
    ('회귀', '출력층: Linear', '손실: MSE'),
]

for task, out_layer, loss in mapping:
    print(f'{task:8s} | {out_layer:28s} | {loss}')
